## Current Progress

- Cleaned dataset
- Features selected
- Isolation forest trained
- Predictions generated
- 339 or 5% of dataset were anomalies

## To do
- Compare predictions with label, attack_type
- Evaluate model performance(IIMT 2641)

In [170]:
import pandas as pd
df=pd.read_csv("../data/cybersecurity_cleaned.csv")
df

,timestamp,src_ip,dst_ip,src_port,dst_port,protocol,bytes_sent,bytes_received,user_agent,url,is_internal_traffic,label,attack_type
0,2025-10-01 00:12:54,18817627165,253240113218,56377,445,0,8029,17204,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://webmail.corp/login?id=385071,0,0,benign
1,2025-10-01 00:23:43,68592643,2127538111,51165,1433,0,676368,2643374,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://portal.example.org/owa/auth/logon.aspx...,0,0,benign
2,2025-10-01 00:27:21,122119194175,17514078230,36097,443,0,70933,21935,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://webmail.corp/phpmyadmin?id=114701,0,0,benign
3,2025-10-01 00:40:09,18119924268,559917769,445,21255,0,12721,9939,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://portal.example.org/config.php?id=345569,0,0,benign
4,2025-10-01 01:19:57,238229195174,161827224,25167,443,0,3592,1081,Mozilla/5.0 (iPhone; CPU iPhone OS 16_6 like M...,https://internal.bank.local/?id=432022,1,0,benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6763,2025-12-29 22:35:29,1003151108,2491798177,44655,3306,0,102224,161909,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,https://webmail.corp/admin?id=667076,0,0,benign
6764,2025-12-29 22:51:57,152953114,236104126125,3306,22,1,244511,280085,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://webmail.corp/phpmyadmin?id=120556,1,0,benign
6765,2025-12-29 23:03:05,21354194229,2163437197,3389,445,0,10597,16670,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,https://app.company.in/api/v1/users?id=525850,0,0,benign
6766,2025-12-29 23:20:46,1912213100,243249117224,31882,80,0,6062,3504,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,https://app.company.in/manager/html?id=497838,0,0,benign


In [171]:
df["timestamp"]=df["timestamp"].astype("datetime64[s]")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6768 entries, 0 to 6767
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype        
---  ------               --------------  -----        
 0   timestamp            6768 non-null   datetime64[s]
 1   src_ip               6768 non-null   int64        
 2   dst_ip               6768 non-null   int64        
 3   src_port             6768 non-null   int64        
 4   dst_port             6768 non-null   int64        
 5   protocol             6768 non-null   int64        
 6   bytes_sent           6768 non-null   int64        
 7   bytes_received       6768 non-null   int64        
 8   user_agent           6768 non-null   str          
 9   url                  6768 non-null   str          
 10  is_internal_traffic  6768 non-null   int64        
 11  label                6768 non-null   int64        
 12  attack_type          6768 non-null   str          
dtypes: datetime64[s](1), int64(9), str(3)
memory usage: 687.5 K

In [172]:
features = ["src_port", "dst_port", "protocol", "bytes_sent", "bytes_received", "is_internal_traffic"]
X = df[features]

In [173]:
from sklearn.ensemble import IsolationForest
iso_forest = IsolationForest(max_samples=128,n_estimators=100,contamination=0.50, random_state=17)
iso_forest.fit(X)
predictions = iso_forest.predict(X)
predictions

array([ 1, -1,  1, ...,  1,  1, -1], shape=(6768,))

In [174]:
#to answer how many anomalies did the model find?
count = 0
for i in predictions:
    if i==-1:
        count += 1
count

3384

In [175]:
df["prediction"] = predictions
df

,timestamp,src_ip,dst_ip,src_port,dst_port,protocol,bytes_sent,bytes_received,user_agent,url,is_internal_traffic,label,attack_type,prediction
0,2025-10-01 00:12:54,18817627165,253240113218,56377,445,0,8029,17204,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://webmail.corp/login?id=385071,0,0,benign,1
1,2025-10-01 00:23:43,68592643,2127538111,51165,1433,0,676368,2643374,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://portal.example.org/owa/auth/logon.aspx...,0,0,benign,-1
2,2025-10-01 00:27:21,122119194175,17514078230,36097,443,0,70933,21935,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://webmail.corp/phpmyadmin?id=114701,0,0,benign,1
3,2025-10-01 00:40:09,18119924268,559917769,445,21255,0,12721,9939,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://portal.example.org/config.php?id=345569,0,0,benign,-1
4,2025-10-01 01:19:57,238229195174,161827224,25167,443,0,3592,1081,Mozilla/5.0 (iPhone; CPU iPhone OS 16_6 like M...,https://internal.bank.local/?id=432022,1,0,benign,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6763,2025-12-29 22:35:29,1003151108,2491798177,44655,3306,0,102224,161909,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,https://webmail.corp/admin?id=667076,0,0,benign,-1
6764,2025-12-29 22:51:57,152953114,236104126125,3306,22,1,244511,280085,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://webmail.corp/phpmyadmin?id=120556,1,0,benign,-1
6765,2025-12-29 23:03:05,21354194229,2163437197,3389,445,0,10597,16670,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,https://app.company.in/api/v1/users?id=525850,0,0,benign,1
6766,2025-12-29 23:20:46,1912213100,243249117224,31882,80,0,6062,3504,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,https://app.company.in/manager/html?id=497838,0,0,benign,1


In [176]:
anomalies = df[df["prediction"] == -1]
anomalies

,timestamp,src_ip,dst_ip,src_port,dst_port,protocol,bytes_sent,bytes_received,user_agent,url,is_internal_traffic,label,attack_type,prediction
1,2025-10-01 00:23:43,68592643,2127538111,51165,1433,0,676368,2643374,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://portal.example.org/owa/auth/logon.aspx...,0,0,benign,-1
3,2025-10-01 00:40:09,18119924268,559917769,445,21255,0,12721,9939,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://portal.example.org/config.php?id=345569,0,0,benign,-1
4,2025-10-01 01:19:57,238229195174,161827224,25167,443,0,3592,1081,Mozilla/5.0 (iPhone; CPU iPhone OS 16_6 like M...,https://internal.bank.local/?id=432022,1,0,benign,-1
7,2025-10-01 01:57:54,6950130168,20251199195,12037,80,1,136311,418831,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,https://api.service.io/owa/auth/logon.aspx?id=...,1,0,benign,-1
8,2025-10-01 03:38:42,115179112186,23684149193,37290,1433,0,405720,1305168,Mozilla/5.0 (compatible; Googlebot/2.1; +http:...,https://api.service.io/wp-login.php?id=407810,0,0,benign,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6760,2025-12-29 21:33:20,23018038196,143671476,64643,443,0,3245,8773,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://api.service.io/owa/auth/logon.aspx?id=...,1,0,benign,-1
6761,2025-12-29 21:49:44,215530131,161237197247,22,21,1,260,88,curl/8.4.0,https://webmail.corp/test.php?id=694357,0,0,benign,-1
6763,2025-12-29 22:35:29,1003151108,2491798177,44655,3306,0,102224,161909,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,https://webmail.corp/admin?id=667076,0,0,benign,-1
6764,2025-12-29 22:51:57,152953114,236104126125,3306,22,1,244511,280085,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,https://webmail.corp/phpmyadmin?id=120556,1,0,benign,-1


In [177]:
#Out of the predicted anomalies, how many are 0, how many are 1?
anomalies["label"].value_counts()

label
0    3258
1     126
Name: count, dtype: int64

In [178]:
anomalies["attack_type"].value_counts()

attack_type
benign                 3258
brute-force              27
sql-injection            23
port-scan                22
ddos                     18
xss                      13
exploit-attempt           9
c2                        5
credential-stuffing       5
command-injection         4
Name: count, dtype: int64

In [179]:
df["label"].value_counts()

label
0    6499
1     269
Name: count, dtype: int64

In [180]:
df["attack_type"].value_counts()

attack_type
benign                 6499
brute-force              75
port-scan                50
sql-injection            43
xss                      24
ddos                     19
credential-stuffing      17
command-injection        17
exploit-attempt          17
c2                        7
Name: count, dtype: int64